# ISS Group 24 — Two-Model Few-Shot System

**Models**:
1. **Localizer** — multi-shot OWLv2 + learned support fusion + abstain channel + log-scale box head.
   Headline metric: **mAP@50** (positive episodes) + **abstain rate** (negative episodes).
2. **Siamese** — multi-shot DINOv2-small + cross-attention head. Predicts `existence_prob ∈ [0, 1]`.
   Headline metric: **AUROC** + **f1 @ learned threshold** (FPR closely watched).

The two models are **fully independent** — train, evaluate, save, and run inference on either alone.
Combined cascaded inference is in `inference_combined.py`.

**Stages** (every (epoch, fold) checkpoint is resumable):
- Localizer: **Phase 0** (one-shot vanilla OWLv2 baseline) → **L1** (fusion only) → **L2** (+ heads) → **L3** (+ LoRA).
- Siamese:   **Phase 0** (zero-shot DINOv2 cosine baseline) → **S1** (head only) → **S2** (+ LoRA).

Each stage runs an internal **curriculum** (`insdet` → `hots` → `mixed`); the trainer prints which sub-phase it's in.

**Manifest v6**: InsDet support images are physically pre-cropped on disk using **briaai/RMBG-2.0** background-removal masks (tight rectangular bbox + 10% pad). HOTS supports are object-centred at source. The pipeline assumes every support image is object-only — there is no runtime bbox-crop augmentation.

**Threshold policy**: the siamese trainer persists the median val `best_f1_threshold` into `stage_complete.pt`. `evaluate_run` uses that learned threshold by default, fixing the previous `tp=0` cosmetic issue at the hardcoded 0.5 threshold.

**Confusion-matrix data**: every siamese eval JSON contains a `confusion_matrix.records` field with per-episode (instance_id, source, k, score, is_present) tuples so curves can be reconstructed offline.

Set `RUN_SMOKE_TEST = True` to run the ~60s end-to-end smoke check before any training.


In [1]:
import os
import sys
import shutil
import subprocess
import time
from pathlib import Path

In [2]:
SHARED_FOLDER_LINK = "https://drive.google.com/drive/folders/19el_ISdRf5WoXarBsUOXjxFY9A4cPGYJ?usp=sharing"
REPO_URL          = "https://github.com/pewpewnor/iss_group_24.git"
REPO_LOCAL_PATH   = "/content/iss_group_24_src"
USE_GOOGLE_COLAB  = False

# When USE_GOOGLE_COLAB is True we copy the staged dataset from Drive into
# the Colab runtime's local SSD for fast random reads. Checkpoints + analysis
# JSONs always go to the Drive folder so they survive runtime resets.
LOCAL_DATA_ROOT = "/content/iss_group_24_data"


In [3]:
if USE_GOOGLE_COLAB:
    from google.colab import drive

    def mount_drive() -> str:
        drive.mount("/content/drive")
        for candidate in [
            "/content/drive/Shareddrives/iss_group_24",
            "/content/drive/MyDrive/iss_group_24",
        ]:
            if Path(candidate).exists():
                print(f"Drive mounted. Project root: {candidate}")
                return candidate
        raise RuntimeError("iss_group_24 not found on the mounted Drive")

    DRIVE_ROOT = mount_drive()
else:
    DRIVE_ROOT = str(Path(".").resolve())
    print(f"Local mode. Project root: {DRIVE_ROOT}")

Local mode. Project root: /mnt/Onetera/iss_group_24


In [4]:
def setup_repo() -> None:
    if Path(REPO_LOCAL_PATH).exists():
        subprocess.run(["git", "-C", REPO_LOCAL_PATH, "fetch", "origin"], check=True)
        subprocess.run(["git", "-C", REPO_LOCAL_PATH, "reset", "--hard", "origin/main"], check=True)
    else:
        subprocess.run(["git", "clone", "--depth=1", REPO_URL, REPO_LOCAL_PATH], check=True)
    if REPO_LOCAL_PATH not in sys.path:
        sys.path.insert(0, REPO_LOCAL_PATH)


def install_deps() -> None:
    subprocess.run(["pip", "install", "-q",
                    "transformers>=4.40", "accelerate>=0.30", "peft>=0.10",
                    "huggingface_hub>=0.23", "matplotlib>=3.7"], check=True)


if USE_GOOGLE_COLAB:
    setup_repo()
    install_deps()

In [5]:
# ── Path layout ──────────────────────────────────────────────────────────
# DRIVE_ROOT  : project root on Google Drive (or local repo when not on Colab).
# DATA_ROOT   : where the staged dataset lives at READ time. On Colab this
#               points at the local SSD copy after the next cell mirrors it
#               from Drive (fast random reads). Off Colab it's the Drive copy.
# OUT_ROOT    : where checkpoints are written (always on Drive — survives
#               Colab runtime resets).
# ANALYSIS_ROOT : where per-(epoch, fold) JSON + plots are written (always on Drive).
DRIVE_DATA_ROOT     = f"{DRIVE_ROOT}/dataset/aggregated"
DRIVE_MANIFEST_PATH = f"{DRIVE_DATA_ROOT}/manifest.json"

OUT_ROOT      = f"{DRIVE_ROOT}/checkpoints"
ANALYSIS_ROOT = f"{DRIVE_ROOT}/analysis"

# These two get re-pointed in the next cell when USE_GOOGLE_COLAB is True.
DATA_ROOT = DRIVE_DATA_ROOT
MANIFEST  = DRIVE_MANIFEST_PATH

os.chdir(DRIVE_ROOT)
print(f"Working directory: {os.getcwd()}")
print(f"DRIVE_ROOT     = {DRIVE_ROOT}")
print(f"DATA_ROOT      = {DATA_ROOT}")
print(f"OUT_ROOT       = {OUT_ROOT}")
print(f"ANALYSIS_ROOT  = {ANALYSIS_ROOT}")

# Hard guard: when on Colab, refuse to start training unless OUT_ROOT and
# ANALYSIS_ROOT are on Drive (so every per-(epoch, fold) checkpoint survives a
# runtime reset).  This catches misconfigured paths early.
from shared.checkpoint import assert_checkpoint_root_on_drive
assert_checkpoint_root_on_drive(OUT_ROOT,      on_colab=USE_GOOGLE_COLAB)
assert_checkpoint_root_on_drive(ANALYSIS_ROOT, on_colab=USE_GOOGLE_COLAB)


Working directory: /mnt/Onetera/iss_group_24
DRIVE_ROOT     = /mnt/Onetera/iss_group_24
DATA_ROOT      = /mnt/Onetera/iss_group_24/dataset/aggregated
OUT_ROOT       = /mnt/Onetera/iss_group_24/checkpoints
ANALYSIS_ROOT  = /mnt/Onetera/iss_group_24/analysis


---
## Step 0 — Build dataset manifest (writes to Drive)

Stages HOTS + InsDet → `{DRIVE_ROOT}/dataset/aggregated/{train,test}`.
**Idempotent**: re-runs only when staging is missing or invalid.

**Manifest v6**: when (re-)building, InsDet supports are physically cropped to the tight RMBG-2.0 foreground bbox + 10% pad. This takes ~1 minute on GPU for ~2400 supports. Use `aggregator.main(force=True)` to rebuild from scratch.

Note: this writes to Drive even on Colab — Drive is the source of truth for the staged dataset and we mirror it locally in the next cell for speed.


In [6]:
import aggregator
aggregator.main()

validate: checked 9083 (path, bbox) pairs, 0 bad
aggregator: dataset already staged at dataset/aggregated and validates; skipping.


---
## Step 0b — (Colab only) Mirror dataset to local runtime SSD

Reads `{DRIVE_ROOT}/dataset/aggregated/` and copies it to `LOCAL_DATA_ROOT`
on the Colab runtime, with progress printed. Drive random-read latency is
~50-100 ms per file; copying once and reading from local SSD takes the
per-file latency down to <1 ms, easily 50× faster training.

After the copy, `DATA_ROOT` and `MANIFEST` are re-pointed to the local copy.
**`OUT_ROOT` and `ANALYSIS_ROOT` continue to point to Drive** so checkpoints
and analysis JSONs persist across runtime resets.

If anything goes wrong (out of disk, permission, etc.) we fall back to reading
directly from Drive.


In [7]:
def mirror_to_local(src: str, dst: str, *, progress_every: float = 2.0) -> bool:
    """Copy a directory tree from src → dst with periodic progress prints.

    Returns True on success, False otherwise. Falls back to no-op when
    src and dst resolve to the same path.

    Skips files that already exist at dst with the same size + mtime, so
    re-running is cheap.
    """
    src_p = Path(src).resolve()
    dst_p = Path(dst).resolve()
    if src_p == dst_p:
        print(f"  src == dst ({src_p}); nothing to do.")
        return True
    if not src_p.exists():
        print(f"  ✗ source not found: {src_p}")
        return False

    # First pass: enumerate.
    files: list[tuple[Path, Path]] = []
    total_bytes = 0
    for p in src_p.rglob("*"):
        if p.is_file():
            rel = p.relative_to(src_p)
            target = dst_p / rel
            files.append((p, target))
            try:
                total_bytes += p.stat().st_size
            except OSError:
                pass
    n_files = len(files)
    print(f"  found {n_files} files ({total_bytes / 1024 / 1024:.1f} MB) under {src_p}")

    dst_p.mkdir(parents=True, exist_ok=True)
    n_copied = 0
    n_skipped = 0
    bytes_copied = 0
    t_start = time.time()
    t_last = t_start
    for s, d in files:
        try:
            d.parent.mkdir(parents=True, exist_ok=True)
            if d.exists():
                try:
                    s_st = s.stat()
                    d_st = d.stat()
                    if s_st.st_size == d_st.st_size and abs(s_st.st_mtime - d_st.st_mtime) < 1:
                        n_skipped += 1
                        n_copied += 1
                        bytes_copied += s_st.st_size
                        continue
                except OSError:
                    pass
            shutil.copy2(str(s), str(d))
            try:
                bytes_copied += s.stat().st_size
            except OSError:
                pass
            n_copied += 1
        except Exception as e:                                                # noqa: BLE001
            print(f"  ✗ failed to copy {s} → {d}: {e}")
            return False
        now = time.time()
        if now - t_last >= progress_every or n_copied == n_files:
            elapsed = now - t_start
            mb_done = bytes_copied / 1024 / 1024
            mb_total = total_bytes / 1024 / 1024
            rate = mb_done / max(elapsed, 1e-6)
            pct = 100.0 * n_copied / max(n_files, 1)
            eta = (mb_total - mb_done) / max(rate, 1e-6)
            print(
                f"  [{n_copied}/{n_files} = {pct:5.1f}%]  "
                f"{mb_done:7.1f} / {mb_total:7.1f} MB  "
                f"({rate:6.1f} MB/s)  elapsed={elapsed:5.1f}s  eta={eta:5.1f}s",
                flush=True,
            )
            t_last = now
    print(f"  ✓ copied {n_copied} files ({n_skipped} unchanged) in {time.time() - t_start:.1f}s")
    return True


if USE_GOOGLE_COLAB:
    print(f"Mirroring {DRIVE_DATA_ROOT} → {LOCAL_DATA_ROOT} …")
    ok = mirror_to_local(DRIVE_DATA_ROOT, LOCAL_DATA_ROOT)
    if ok:
        DATA_ROOT = LOCAL_DATA_ROOT
        MANIFEST  = f"{LOCAL_DATA_ROOT}/manifest.json"
        print(f"  ✓ DATA_ROOT switched to local: {DATA_ROOT}")
        print(f"  ✓ MANIFEST  switched to local: {MANIFEST}")
    else:
        print("  ⚠ mirror failed; falling back to Drive paths.")
        DATA_ROOT = DRIVE_DATA_ROOT
        MANIFEST  = DRIVE_MANIFEST_PATH
else:
    print("Not on Colab — DATA_ROOT stays at the project's local copy.")

print()
print(f"DATA_ROOT      = {DATA_ROOT}")
print(f"MANIFEST       = {MANIFEST}")
print(f"OUT_ROOT       = {OUT_ROOT}    (always Drive on Colab)")
print(f"ANALYSIS_ROOT  = {ANALYSIS_ROOT}    (always Drive on Colab)")


Not on Colab — DATA_ROOT stays at the project's local copy.

DATA_ROOT      = /mnt/Onetera/iss_group_24/dataset/aggregated
MANIFEST       = /mnt/Onetera/iss_group_24/dataset/aggregated/manifest.json
OUT_ROOT       = /mnt/Onetera/iss_group_24/checkpoints    (always Drive on Colab)
ANALYSIS_ROOT  = /mnt/Onetera/iss_group_24/analysis    (always Drive on Colab)


---
## Step 1 — Smoke test (~60s)

End-to-end check: aggregator validate → localizer L1/L2/L3 → siamese S1/S2 →
combined inference → plots → checkpoint roundtrip. Smoke artifacts go to
`{OUT_ROOT}/_smoke/` and `{ANALYSIS_ROOT}/_smoke/` and are wiped at the end.

Set `RUN_SMOKE_TEST = False` to skip. Recommended on first run.


In [8]:
RUN_SMOKE_TEST = False

if RUN_SMOKE_TEST:
    from shared.smoke import smoke_test
    res = smoke_test(seconds_budget=60.0, manifest=MANIFEST, data_root=DATA_ROOT)
    print(f"smoke status={res['status']}  wall_clock={res['wall_clock_seconds']:.1f}s")

---
## Step 2 — Imports + shared kwargs

Every parameter in `SHARED_KWARGS` is overridable per-stage. The localizer and
siamese pull defaults from their own `DEFAULT_CFG`; entries below are the most
common knobs you might want to tune.

If a training cell is interrupted (Ctrl-C / kernel-interrupt), the trainer
catches the interrupt and frees CUDA VRAM before re-raising — no kernel
restart needed.


In [ ]:
import torch
from localizer.train import (
    train_stage_L1  as loc_train_L1,
    train_stage_L2  as loc_train_L2,
    train_stage_L3  as loc_train_L3,
    evaluate_phase0 as loc_evaluate_phase0,
    evaluate_run    as loc_evaluate_run,
)
from siamese.train import (
    train_stage_S1  as sia_train_S1,
    train_stage_S2  as sia_train_S2,
    evaluate_phase0 as sia_evaluate_phase0,
    evaluate_run    as sia_evaluate_run,
)
from inference_combined import run_combined, sweep_threshold
from shared.plots import plot_all_from_jsons
from shared.runtime import release_gpu_memory


In [10]:
# VRAM-aware config tier.
_VRAM = torch.cuda.get_device_properties(0).total_memory if torch.cuda.is_available() else 0
_HAS_BIG_GPU = _VRAM >= 16e9
if _HAS_BIG_GPU:
    LOC_IMG, LOC_BATCH, LOC_ACCUM = 960, 4, 2
    SIA_IMG, SIA_BATCH, SIA_ACCUM = 518, 8, 1
elif _VRAM >= 5.5e9:
    LOC_IMG, LOC_BATCH, LOC_ACCUM = 768, 1, 8
    SIA_IMG, SIA_BATCH, SIA_ACCUM = 518, 4, 2
else:
    LOC_IMG, LOC_BATCH, LOC_ACCUM = 768, 1, 8
    SIA_IMG, SIA_BATCH, SIA_ACCUM = 518, 2, 4
print(f"VRAM={_VRAM/1e9:.1f} GB | localizer img={LOC_IMG} B={LOC_BATCH} accum={LOC_ACCUM} | siamese img={SIA_IMG} B={SIA_BATCH} accum={SIA_ACCUM}")

VRAM=6.0 GB | localizer img=768 B=1 accum=8 | siamese img=518 B=4 accum=2


In [ ]:
# Shared paths/IO/folds — every key is forwardable to both trainers.
SHARED_KWARGS = dict(
    manifest      = MANIFEST,       # local SSD on Colab; Drive elsewhere
    data_root     = DATA_ROOT,      # local SSD on Colab; Drive elsewhere
    out_root      = OUT_ROOT,       # always Drive
    analysis_root = ANALYSIS_ROOT,  # always Drive
    num_workers   = 2,
    use_amp       = True,
    seed          = 42,
    folds         = 3,
    fold_seed     = 42,
    k_min         = 1,
    k_max         = 10,
    keep_last_n   = 0,             # keep every per-(epoch, fold) ckpt
)

# Localizer-specific overrides.
#
# Stage roles after rebalance:
#   L1  fusion-only warm-up (4 ep total). Frozen OWLv2 + heads. Positive-only.
#   L2  short head warm-up (4 ep total) BEFORE L3. Unfreezes class+box heads
#       and runs them at the HIGHEST LR in the schedule so they snap into
#       the fused prototype. NO LoRA yet — backbone still frozen.
#   L3  main fine-tune (12 ep total). Attaches LoRA. Drops fusion+head LRs
#       4-5× vs L2 and adds LoRA at 2e-4 (fresh init). This is where mAP
#       actually moves; the previous L1/L2/L3 had L2 doing the heavy lifting.
LOC_KWARGS = dict(
    **SHARED_KWARGS,
    img_size              = LOC_IMG,
    batch_size            = LOC_BATCH,
    grad_accum_steps      = LOC_ACCUM,
    # ── Curriculum (per stage). Set a phase's epoch count to 0 to skip it.
    L1_curriculum         = ['insdet', 'hots', 'mixed'],
    L2_curriculum         = ['insdet', 'hots', 'mixed'],
    L3_curriculum         = ['insdet', 'hots', 'mixed'],
    L1_epochs_insdet      = 1, L1_epochs_hots = 1, L1_epochs_mixed = 2,
    L2_epochs_insdet      = 1, L2_epochs_hots = 1, L2_epochs_mixed = 2,
    L3_epochs_insdet      = 3, L3_epochs_hots = 3, L3_epochs_mixed = 6,
    L1_eps_per_fold       = 400,
    L2_eps_per_fold       = 300,
    L3_eps_per_fold       = 300,
    val_episodes          = 100,
    test_episodes         = 400,
    # ── Per-stage negative-episode rates.
    L1_neg_prob           = 0.0,   # positive-only at L1
    L2_neg_prob           = 0.25,
    L3_neg_prob           = 0.30,
    # ── LRs. L2 is a WARM-UP with the biggest LRs; L3 drops everything
    #    4-5× and adds LoRA on top at 2e-4.
    lr_fusion_L1          = 2e-4,
    lr_fusion_L2          = 3e-4,
    lr_class_L2           = 1e-4,
    lr_box_L2             = 1e-4,
    lr_fusion_L3          = 5e-5,
    lr_class_L3           = 2e-5,
    lr_box_L3             = 2e-5,
    lr_lora_L3            = 2e-4,  # LoRA params are fresh-init; need higher LR
    # ── Loss weighting (GIoU > L1 > log-area regulariser).
    lambda_patch_ce       = 1.0,
    lambda_l1             = 2.0,
    lambda_giou           = 4.0,
    lambda_log_area       = 0.3,
    patch_ce_label_smoothing  = 0.05,
    patch_ce_neighbour_radius = 1,
    patch_ce_neighbour_weight = 0.30,
    L2_box_warmup_epochs  = 0,     # L2 is itself a warm-up; no inner freeze.
    fusion_layers         = 2,
    fusion_heads          = 8,
    fusion_dropout        = 0.0,
    lora_r                = 8,
    lora_alpha            = 16,
    lora_dropout          = 0.1,
    lora_last_n_layers    = 4,
    abstain_threshold     = 0.5,
    L1_early_stop_patience = 3,
    L2_early_stop_patience = 3,
    L3_early_stop_patience = 5,
)

# Siamese-specific overrides.
#
# Stage roles:
#   S1  head-only training (10 ep total). DINOv2 frozen. Highest head LR.
#   S2  + LoRA fine-tune (6 ep total). Head LR drops 5×; LoRA gets a fresh-
#       init LR roughly between S1 and S2 head LRs.
#
# focal_alpha=0.5 (balanced) + neg_prob=0.5 (1:1). Per-epoch val metrics are
# reported at the val's own best_f1_threshold (AUTO mode), and the median of
# those thresholds is persisted into stage_complete.pt so evaluate_run picks
# a calibrated threshold at test time.
SIA_KWARGS = dict(
    **SHARED_KWARGS,
    img_size              = SIA_IMG,
    batch_size            = SIA_BATCH,
    grad_accum_steps      = SIA_ACCUM,
    S1_curriculum         = ['insdet', 'hots', 'mixed'],
    S2_curriculum         = ['insdet', 'hots', 'mixed'],
    S1_epochs_insdet      = 2, S1_epochs_hots = 2, S1_epochs_mixed = 6,
    S2_epochs_insdet      = 1, S2_epochs_hots = 1, S2_epochs_mixed = 4,
    S1_eps_per_fold       = 500,
    S2_eps_per_fold       = 400,
    val_episodes          = 200,
    test_episodes         = 400,
    # ── LRs.
    lr_head_S1            = 1e-3,
    lr_head_S2            = 2e-4,
    lr_lora_S2            = 3e-4,
    # ── Loss.
    focal_alpha           = 0.5,
    focal_gamma           = 2.0,
    variance_target       = 0.4,
    variance_weight       = 0.05,
    decorr_weight         = 0.02,
    neg_prob              = 0.5,
    hard_neg_cache_frac   = 0.5,
    # ── Architecture.
    cross_attn_heads      = 6,
    head_hidden_1         = 256,
    head_hidden_2         = 64,
    head_dropout          = 0.2,
    lora_r                = 8,
    lora_alpha            = 16,
    lora_dropout          = 0.1,
    lora_last_n_layers    = 4,
    # ── Eval.
    eval_threshold        = 0.5,    # overridden at eval_run by learned_threshold
    early_stop_metric     = 'f1',
    S1_early_stop_patience = 4,
    S2_early_stop_patience = 3,
)


---
# Localizer

OWLv2 + learnable support fusion + abstain channel + log-scale box head.
Phase 0 is a **one-shot vanilla OWLv2 baseline** (K=1, no fusion, no abstain)
— the numerical floor each trained stage must beat. L1 trains the fusion
stack from a zero-correction warm-start; L2 unfreezes the OWLv2 class +
box heads + adds the log-scale box wrapper; L3 attaches LoRA to the last 4
ViT blocks.

Each stage internally runs a 3-phase curriculum (`insdet` → `hots` → `mixed`).


## Phase 0 — one-shot vanilla OWLv2 baseline

Apples-to-apples reference. The model picks the first valid support per
episode and runs bare OWLv2 image-guided detection (no fusion, no abstain,
no log-scale box head). Eval is forced to **K=1** so this really is the
one-shot baseline.


In [ ]:
loc_evaluate_phase0(**LOC_KWARGS)

## Stage L1 — fusion-only warm-up (positive-only)

Trains only the support-fusion transformer + support-attn-pool + abstain
channel. OWLv2 backbone + class/box heads stay frozen. Positive-only
episodes (no negatives yet — the abstain channel is moot until L2). The
goal is for the fusion to converge to a usable prototype that the L2
warm-up can then start moving the heads against.

**Sizing**: 4 epochs total, distributed `1 insdet + 1 hots + 2 mixed`.
Single LR (`lr_fusion_L1 = 2e-4`).


In [ ]:
loc_train_L1(**LOC_KWARGS)

In [ ]:
loc_evaluate_run(
    checkpoint=f'{OUT_ROOT}/localizer/L1/stage_complete.pt',
    **LOC_KWARGS,
)

## Stage L2 — short head warm-up before L3

Unfreezes OWLv2's class_head, box_head and final layer_norm. The point
of L2 is to let the OWLv2 heads + the new log-scale box wrapper *snap
into* the fused prototype that L1 produced, before L3 starts moving
the backbone via LoRA. Backbone stays FROZEN; NO LoRA here.

**Sizing**: 4 epochs total, `1 insdet + 1 hots + 2 mixed`. LRs are the
highest in the schedule (`fusion 3e-4, heads 1e-4`) precisely because
this stage is short and exists only to converge the heads quickly.
Mixed pos+neg (`neg_prob=0.25`) so the abstain channel starts learning.


In [ ]:
loc_train_L2(**LOC_KWARGS)

In [ ]:
loc_evaluate_run(
    checkpoint=f'{OUT_ROOT}/localizer/L2/stage_complete.pt',
    **LOC_KWARGS,
)

## Stage L3 — main fine-tune (+ LoRA on the last 4 OWLv2 ViT blocks)

This is the longest stage and where actual mAP gains come from. Attaches
LoRA (r=8, α=16) on `q_proj` / `v_proj` of the last 4 vision blocks. The
fusion + heads have already converged in L1/L2 so their LRs drop 4-5×
vs L2 (`fusion 5e-5, heads 2e-5`). LoRA is fresh-init and gets the
highest LR in L3 (`2e-4`).

**Sizing**: 12 epochs total, `3 insdet + 3 hots + 6 mixed`. Slightly
higher negative rate (`neg_prob=0.30`) so the abstain channel keeps
improving while the backbone moves.


In [ ]:
loc_train_L3(**LOC_KWARGS)

In [ ]:
loc_evaluate_run(
    checkpoint=f'{OUT_ROOT}/localizer/L3/stage_complete.pt',
    **LOC_KWARGS,
)

---
# Siamese

Frozen DINOv2-small backbone + cross-attention pooling head. Predicts
`existence_prob ∈ [0, 1]`. Trained with **focal-BCE (α=0.5, γ=2)** + variance
+ decorrelation regularisers; positive : negative ≈ 1 : 1 (`neg_prob=0.5`).

**Threshold policy (NEW)**: per-epoch val metrics are reported at the val's
own `best_f1_threshold` (AUTO mode), and the median of those thresholds is
persisted into `stage_complete.pt` as `learned_threshold`. `evaluate_run`
loads it automatically — no more `tp=0` because the 0.5 threshold was
never crossed.

**Confusion-matrix data (NEW)**: every eval JSON now contains a
`confusion_matrix.records` list of (instance_id, source, k, score, is_present)
tuples so curves and operating-point sweeps can be rebuilt offline.


## Phase 0 — zero-shot DINOv2 cosine baseline

Cosine similarity of the mean support CLS-token embedding vs the query
CLS-token, then `prob = (cos + 1) / 2`. Evaluated at AUTO threshold so
the baseline's precision/recall are reported at *its* best-F1 operating
point (apples-to-apples with the trained S1/S2 reports).


In [ ]:
sia_evaluate_phase0(**SIA_KWARGS)

## Stage S1 — head-only training (DINOv2 frozen)

Trains the cross-attention pool + scalar-feature MLP head on top of a
frozen DINOv2-small. Highest head LR in the schedule (`1e-3`) — the
head is from-scratch.

**Sizing**: 10 epochs total, biased toward mixed (`2 insdet + 2 hots
+ 6 mixed`).


In [ ]:
sia_train_S1(**SIA_KWARGS)

In [ ]:
sia_evaluate_run(
    checkpoint=f'{OUT_ROOT}/siamese/S1/stage_complete.pt',
    **SIA_KWARGS,
)

## Stage S2 — + LoRA fine-tune on the last 4 DINOv2 blocks

Attaches LoRA (r=8, α=16) on `query` / `value` of the last 4 DINOv2
encoder layers. Head LR drops 5× (`2e-4`) because the head is already
converged from S1. LoRA gets its own fresh-init LR (`3e-4`).

**Sizing**: 6 epochs total, `1 insdet + 1 hots + 4 mixed`.


In [ ]:
sia_train_S2(**SIA_KWARGS)

In [ ]:
sia_evaluate_run(
    checkpoint=f'{OUT_ROOT}/siamese/S2/stage_complete.pt',
    **SIA_KWARGS,
)

---
# Combined inference & threshold sweep

Cascaded siamese → localizer. The siamese decides `exists`; if `True`,
the localizer runs to produce a bbox. `run_combined(existence_threshold=None)`
(default) reads the siamese checkpoint's persisted `learned_threshold`.


## Threshold sweep on the test split

In [ ]:
sweep_threshold(
    siamese_ckpt   = f'{OUT_ROOT}/siamese/S2/stage_complete.pt',
    localizer_ckpt = f'{OUT_ROOT}/localizer/L3/stage_complete.pt',
    manifest       = MANIFEST,
    data_root      = DATA_ROOT,
    test_episodes  = 400,
    thresholds     = (0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60, 0.65, 0.70),
    siamese_img_size   = SIA_IMG,
    localizer_img_size = LOC_IMG,
    batch_size     = 1,
    num_workers    = 0,
    seed           = 42,
    k_min          = 1,
    k_max          = 10,
    analysis_root  = ANALYSIS_ROOT,
)

## Single-pair combined inference example

In [ ]:
# Example — uncomment and replace paths.
# run_combined(
#     siamese_ckpt   = f'{OUT_ROOT}/siamese/S2/stage_complete.pt',
#     localizer_ckpt = f'{OUT_ROOT}/localizer/L3/stage_complete.pt',
#     support_paths  = ['path/to/support_01.jpg', 'path/to/support_02.jpg'],
#     query_path     = 'path/to/scene.jpg',
#     siamese_img_size   = SIA_IMG,
#     localizer_img_size = LOC_IMG,
#     existence_threshold      = None,  # None ⇒ use siamese ckpt's learned_threshold
#     existence_threshold_mode = 'hard',
#     abstain_threshold        = 0.5,
# )

---
# Plots

Reads per-(epoch, fold) JSON under `{ANALYSIS_ROOT}` and writes per-stage
training curves to `{ANALYSIS_ROOT}/plots/`.


In [ ]:
plot_all_from_jsons(ANALYSIS_ROOT)

## Free GPU VRAM

Manual reclaim after the trainer's `gpu_cleanup_on_exit` (in case some
lingering frames are still holding tensors).


In [ ]:
release_gpu_memory()

In [ ]:
!nvidia-smi